# CareCompass: MedGemma 4B LoRA for the triage classifier

Teaches MedGemma the exact production task: **CaseFacts JSON in, Decision JSON out** (`app/triage.py`).
Data comes from `training/build_sft.py` (`train.jsonl`, `val.jsonl`, `test.jsonl`; chat format, system prompt = `triage.RULES`).

**Kaggle setup:** Settings -> Accelerator **GPU T4 x1** (or P100), Internet **on**. Add the three JSONL files as a Kaggle Dataset
(e.g. `carecompass-sft`). Colab: upload them next to the notebook. Training takes roughly 20-40 min on ~1.2k rows.

Lines marked **VERIFY** use library APIs that change often (Unsloth / TRL). If a cell fails, compare it with Unsloth's current
Gemma 3 notebook: MedGemma 4B is a Gemma 3 4B fine-tune, so their Gemma 3 recipe applies.

MedGemma is under Google's Health AI Developer Foundations terms; the fine-tune inherits them.

In [ ]:
%%capture
# VERIFY: Unsloth's install line for Kaggle/Colab changes between releases; copy theirs if this fails.
!pip install -q unsloth

In [ ]:
import glob, json, os
hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True) or glob.glob("train.jsonl")
DATA = os.path.dirname(hits[0]) or "."
print("data in", DATA, {f: sum(1 for _ in open(f"{DATA}/{f}.jsonl", encoding="utf-8")) for f in ("train", "val", "test")})

## 1. Model + LoRA
Language layers only: the vision tower is never used (MedGemma only ever sees text facts).
`r=16` is plenty for a narrow 6-label task; raise it to 32 only if val loss plateaus high.

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/medgemma-4b-it",   # VERIFY name; not gated on HF. Smaller download: unsloth/medgemma-4b-it-unsloth-bnb-4bit
    max_seq_length=2048,
    load_in_4bit=True,                      # QLoRA: fits a T4 (16 GB) easily
    full_finetuning=False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)

## 2. Data -> Gemma 3 chat text
The Gemma 3 template folds the system prompt into the first user turn, which is what Ollama's MedGemma template does too.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, train_on_responses_only

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
ds = load_dataset("json", data_files={"train": f"{DATA}/train.jsonl", "val": f"{DATA}/val.jsonl"})

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
                     for m in batch["messages"]]}

ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)
print(ds["train"][0]["text"][-700:])
lens = [len(tokenizer(text=t)["input_ids"]) for t in ds["train"]["text"][:200]]
print("tokens per example: max", max(lens), "mean", sum(lens) // len(lens))

## 3. Train (loss on the model's answer only)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,                    # VERIFY: newer TRL calls this processing_class
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,      # effective batch 16
        num_train_epochs=2,                 # ~1.2k rows -> ~150 steps; the best checkpoint by val loss is kept
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps", eval_steps=30,
        save_strategy="steps", save_steps=30, save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        seed=3407, output_dir="outputs", report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")

# sanity: only the JSON answer should be unmasked
ex = trainer.train_dataset[0]
print(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]))

In [ ]:
stats = trainer.train()
print(stats)

## 4. Quick held-out check before exporting
Greedy decoding, like production (temperature 0). The real comparison runs locally with `training/eval_finetune.py`
against the quantised GGUF; this only catches a broken run before you spend time on export.

In [ ]:
FastModel.for_inference(model)
test = [json.loads(l) for l in open(f"{DATA}/test.jsonl", encoding="utf-8")]
ok = valid = em = em_hit = 0
for r in test:
    inputs = tokenizer.apply_chat_template(r["messages"][:2], add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        d = json.loads(text[text.find("{"): text.rfind("}") + 1]); valid += 1
        got = "need_more_info" if d.get("status") == "need_more_info" else d.get("care_type")
    except Exception:
        got = "INVALID"
    ok += got == r["target"]
    if r["target"] == "emergency_112":
        em += 1; em_hit += got == "emergency_112"
    if got != r["target"]:
        print("MISS", r["target"], "->", got, "|", r["messages"][1]["content"][15:140])
print(f"accuracy {ok}/{len(test)}  valid JSON {valid}/{len(test)}  emergency recall {em_hit}/{em}")

## 5. Save
Always save the LoRA adapter first (small, ~100-150 MB): it is the fallback if GGUF export fails.

In [ ]:
model.save_pretrained("medgemma_triage_lora")
tokenizer.save_pretrained("medgemma_triage_lora")
!zip -qr medgemma_triage_lora.zip medgemma_triage_lora && ls -lh medgemma_triage_lora.zip

### GGUF Q4_K_M for Ollama (same quantisation as today's base, so the comparison is fair)
Path A: Unsloth's exporter. Path B: merge to 16-bit and quantise with llama.cpp yourself.
Download only the text-model `.gguf` (~2.5 GB); the vision projector (`mmproj`) is not needed.

In [ ]:
try:
    # VERIFY: FastModel's exporter has taken quantization_type= ("Q8_0", "BF16", "F16"); FastLanguageModel takes quantization_method="q4_k_m"
    model.save_pretrained_gguf("medgemma_triage_gguf", tokenizer, quantization_method="q4_k_m")
except Exception as e:
    print("Path A failed:", e)
    model.save_pretrained_gguf("medgemma_triage_gguf", tokenizer, quantization_type="Q8_0")   # then quantise via Path B
!ls -lh medgemma_triage_gguf

In [ ]:
# Path B (only if A did not produce a Q4_K_M file)
# model.save_pretrained_merged("medgemma_triage_merged", tokenizer)
# !git clone --depth 1 https://github.com/ggml-org/llama.cpp && cd llama.cpp && cmake -B build && cmake --build build -j --target llama-quantize
# !pip install -q -r llama.cpp/requirements.txt
# !python llama.cpp/convert_hf_to_gguf.py medgemma_triage_merged --outtype f16 --outfile medgemma-triage-f16.gguf
# !llama.cpp/build/bin/llama-quantize medgemma-triage-f16.gguf medgemma-triage.Q4_K_M.gguf Q4_K_M

## 6. Back on the laptop
1. Put the `.gguf` in `models/` as `medgemma-triage.Q4_K_M.gguf` (git-ignored) and run
   `ollama create carecompass-medgemma -f models/Modelfile.medgemma-triage`.
2. `python training/eval_finetune.py --model carecompass-medgemma --prompt rules_only --pair`
3. Only if it passes the gates in `training/FINETUNE_PLAN.md`: in `config.yaml` set
   `classifier.model: carecompass-medgemma` and `classifier.prompt: rules_only`, then rerun `eval/eval_dataset.py`.